# Phase 3, experiment 5: a new validation scheme (expanding-window folds)

**Why this change:** The single split used so far (train on steps 1 to 21, validate on 22 to 35) has
turned out to be a poor guide:

- It is noisy: similar models differ by 0.01 to 0.02 (notebook 04).
- It disagreed with the public leaderboard once (notebook 03).
- Its score is dominated by four steps (24, 29, 31, 32) that contain a kind of illicit transaction
  the model never saw in steps 1 to 21 (notebook 05). The submitted models are trained on steps 1 to
  35, so they do see it.

**Three expanding-window folds:**

| Fold | Train on steps | Validate on steps |
|---|---|---|
| 1 | 1 to 14 | 15 to 21 |
| 2 | 1 to 21 | 22 to 28 |
| 3 | 1 to 28 | 29 to 35 |

"Expanding window" means each fold trains on everything up to a point in time and validates on the
next 7 steps, and the training window grows from fold to fold. Every fold still predicts the future
from the past, so there is no leakage across time. Each experiment is now scored on all three folds,
and the average is the headline number.

**What this notebook does:** It re-measures the three feature sets from earlier notebooks (raw,
raw + degree, raw + degree + neighbour means) under the new scheme, with the default LightGBM, so that
the earlier experiments can be compared on the same footing. Nothing is submitted.

## 1. Load

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")     
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")

raw_cols = [c for c in train.columns if c.startswith("feat_")]

## 2. The folds

Each fold is described by the last training step and the last validation step. Validation starts at
the step after the last training step.

In [2]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]

def split_fold(df, fold):
    # Return the (training rows, validation rows) of df for one fold.
    tr = df[df["time_step"] <= fold["train_last"]]
    va = df[(df["time_step"] > fold["train_last"]) & (df["time_step"] <= fold["valid_last"])]
    return tr, va

## 3. Features (same definitions as notebooks 03 and 06)

In [ ]:
#Degree features, counted over all edges
out_deg = edges["txId1"].value_counts()
in_deg = edges["txId2"].value_counts()

train = train.copy()
train["in_degree"] = train["txId"].map(in_deg).fillna(0).astype(int)
train["out_degree"] = train["txId"].map(out_deg).fillna(0).astype(int)
train["total_degree"] = train["in_degree"] + train["out_degree"]
degree_cols = ["in_degree", "out_degree", "total_degree"]

The neighbour means are taken over the 20 most important raw features. In notebook 06 these were
chosen using steps 1 to 21. Here fold 1 validates on steps 15 to 21, so the choice is made from steps
1 to 14 only. That way no fold's validation rows influence which features are used.

In [ ]:
N_TOP = 20
FIRST_TRAIN_LAST = FOLDS[0]["train_last"]   

is_labeled = train["label"].notna()
early = train[is_labeled & (train["time_step"] <= FIRST_TRAIN_LAST)]

selector = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
selector.fit(early[raw_cols], early["label"].astype(int))
importance = pd.Series(selector.feature_importances_, index=raw_cols).sort_values(ascending=False)
top_cols = list(importance.head(N_TOP).index)
print(top_cols)

# Neighbour means
tx_feats = pd.concat([train[["txId"] + top_cols], test[["txId"] + top_cols]])

# Predecessors: average the features of the source transactions of each destination
pred_mean = edges.merge(tx_feats, left_on="txId1", right_on="txId").groupby("txId2")[top_cols].mean()
pred_mean.columns = ["pred_mean_" + c for c in top_cols]

# Successors: average the features of the destination transactions of each source
succ_mean = edges.merge(tx_feats, left_on="txId2", right_on="txId").groupby("txId1")[top_cols].mean()
succ_mean.columns = ["succ_mean_" + c for c in top_cols]

neigh_cols = list(pred_mean.columns) + list(succ_mean.columns)
train = train.merge(pred_mean, left_on="txId", right_index=True, how="left")
train = train.merge(succ_mean, left_on="txId", right_index=True, how="left")

['feat_2', 'feat_58', 'feat_142', 'feat_59', 'feat_53', 'feat_138', 'feat_52', 'feat_8', 'feat_132', 'feat_3', 'feat_88', 'feat_102', 'feat_55', 'feat_64', 'feat_90', 'feat_94', 'feat_89', 'feat_16', 'feat_137', 'feat_76']


In [5]:
# Labeled rows only (1 = illicit, 0 = licit)
labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

feature_sets = {
    "raw": raw_cols,
    "raw+degree": raw_cols + degree_cols,
    "raw+degree+neigh": raw_cols + degree_cols + neigh_cols,
}

# Size of each fold
fold_info = []
for fold in FOLDS:
    tr, va = split_fold(labeled, fold)
    fold_info.append({
        "fold": fold["name"],
        "train_steps": f"1-{fold['train_last']}",
        "valid_steps": f"{fold['train_last'] + 1}-{fold['valid_last']}",
        "train_rows": len(tr),
        "train_illicit": tr["label"].sum(),
        "valid_rows": len(va),
        "valid_illicit": va["label"].sum(),
    })
pd.DataFrame(fold_info).set_index("fold")

,train_steps,valid_steps,train_rows,train_illicit,valid_rows,valid_illicit
fold,,,,,,
fold 1,1-14,15-21,14896,1005,4634,866
fold 2,1-21,22-28,19530,1871,5677,671
fold 3,1-28,29-35,25207,2542,6028,1102


## 4. Score every feature set on every fold

For each fold and feature set, a default LightGBM is trained on the fold's training steps and scored
on its validation steps (3 folds x 3 feature sets = 9 models).

In [6]:
score_rows = []      # one row per (fold, feature set)
per_step_rows = []   # one row per (fold, feature set, validation step)

for fold in FOLDS:
    tr, va = split_fold(labeled, fold)
    for fs_name, cols in feature_sets.items():
        model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
        model.fit(tr[cols], tr["label"])
        va_pred = model.predict_proba(va[cols])[:, 1]

        score_rows.append({
            "fold": fold["name"],
            "features": fs_name,
            "val_auc": roc_auc_score(va["label"], va_pred),
        })

        # AUC for each validation step of this fold
        for step in sorted(va["time_step"].unique()):
            in_step = (va["time_step"] == step).values
            per_step_rows.append({
                "time_step": step,
                "features": fs_name,
                "auc": roc_auc_score(va["label"][in_step], va_pred[in_step]),
            })
        print("done:", fold["name"], "/", fs_name)

scores = pd.DataFrame(score_rows)
per_step = pd.DataFrame(per_step_rows)

done: fold 1 / raw
done: fold 1 / raw+degree
done: fold 1 / raw+degree+neigh
done: fold 2 / raw
done: fold 2 / raw+degree
done: fold 2 / raw+degree+neigh
done: fold 3 / raw
done: fold 3 / raw+degree
done: fold 3 / raw+degree+neigh


## 5. Results

### 5.1 AUC per fold and the average

`mean` is the average over the three folds and is the headline number from now on. `min` is the
worst fold, which shows how badly a model can do when the validation period contains something new.

In [7]:
sets_order = list(feature_sets.keys())

# Rows = feature sets, columns = folds
fold_table = scores.pivot(index="features", columns="fold", values="val_auc").loc[sets_order]
fold_names = [f["name"] for f in FOLDS]
fold_table["mean"] = fold_table[fold_names].mean(axis=1)
fold_table["min"] = fold_table[fold_names].min(axis=1)
fold_table.round(4)

fold,fold 1,fold 2,fold 3,mean,min
features,,,,,
raw,0.9888,0.9572,0.9965,0.9808,0.9572
raw+degree,0.9887,0.9435,0.9962,0.9761,0.9435
raw+degree+neigh,0.9897,0.9448,0.9964,0.9770,0.9448


### 5.2 Change from each added group of features

In [8]:
change = pd.DataFrame({
    "degree vs raw": fold_table.loc["raw+degree"] - fold_table.loc["raw"],
    "neigh vs raw+degree": fold_table.loc["raw+degree+neigh"] - fold_table.loc["raw+degree"],
}).T
change[fold_names + ["mean"]].round(4)

fold,fold 1,fold 2,fold 3,mean
degree vs raw,-0.0001,-0.0138,-0.0002,-0.0047
neigh vs raw+degree,0.0010,0.0013,0.0001,0.0008


### 5.3 AUC for every validation step (raw features)

Steps 15 to 21 come from fold 1, steps 22 to 28 from fold 2 and steps 29 to 35 from fold 3. Each step
is predicted by a model trained on all the steps before its fold.

In [9]:
step_table = per_step.pivot(index="time_step", columns="features", values="auc")[sets_order]
step_table.round(4)

features,raw,raw+degree,raw+degree+neigh
time_step,,,
15,0.9995,0.9994,0.9995
16,0.9919,0.9903,0.9923
17,0.9891,0.9913,0.9960
18,0.9909,0.9897,0.9919
19,0.9903,0.9916,0.9905
20,0.9802,0.9805,0.9794
21,0.9806,0.9767,0.9810
22,0.9991,0.9993,0.9991
23,0.9591,0.9532,0.9527


## Findings

- **How do the three folds compare (which periods are hard)?** Folds 1 and 3 are easy: raw features
  score 0.9888 on steps 15 to 21 and 0.9965 on steps 29 to 35. Fold 2 (steps 22 to 28) is the hard one
  at 0.9572, and most of that comes from a single step: step 24 scores 0.787, with step 23 at 0.959 and
  step 26 at 0.931. Every other step from 15 to 35 scores 0.98 or higher. Steps 29, 31 and 32, which
  were weak under the old split, now score 0.987 to 0.999, because the model has seen steps 22 to 28.
  This confirms that the weak steps were caused by a new illicit pattern that first appears around
  steps 23 to 24, and that the model handles it once it has training examples.
- **Do degree features help under the new scheme?** No. They change folds 1 and 3 by 0.0002 or less
  and lower fold 2 by 0.0138, for a mean change of -0.0047. The fold 2 drop is concentrated in step 24.
- **Do neighbour means help under the new scheme?** No. They add 0.0010, 0.0013 and 0.0001 on the
  three folds, a mean of +0.0008 over raw + degree, and the result is still below raw features alone
  (0.9770 against 0.9808).
- **How does the new headline number compare with the public leaderboard (about 0.947)?** It is
  higher: the mean over folds is about 0.98. So the folds are easier than the test set. Possible
  reasons are that each fold predicts only the next 7 steps while the test set runs up to 14 steps
  beyond the training data, and that the test period may contain illicit behaviour not present in
  steps 1 to 35. Two of the three folds are close to the maximum score, so they have little room to
  show an improvement; fold 2 is the only one that tests how a model copes with something new.
- **Decision: which feature set to carry forward?** Raw features only. Degree and neighbour means did
  not help on any fold or on the public leaderboard, and fewer features is simpler to justify.
